# Updating a fitted Carver

## About this notebook

In this notebook, we edit the bins of a fitted ``ContinuousCarver`` by hand: we round bin boundaries, group two bins, and check where missing values go.

We start from the setup of the [Continuous Regression example](../ContinuousRegression/continuous_regression_example.ipynb), condensed into one cell. There is one change: we remove 10% of the ``AveOccup`` values at random, so that the carver has missing values to place.

## Installation

In [1]:
# %pip install "AutoCarver[jupyter]"

## Fitting and saving a carver

This cell fits the carver and saves it to the system's temporary folder.

In [2]:
import tempfile
from pathlib import Path

import numpy as np
from sklearn import datasets
from sklearn.model_selection import train_test_split

from AutoCarver import ContinuousCarver, Features

# California Housing data, as in the Continuous Regression example
housing = datasets.fetch_california_housing(as_frame=True)
housing_data = housing["data"]
target = housing["target_names"][0]
housing_data[target] = housing["target"]

# removing 10% of the AveOccup values at random
housing_data.loc[housing_data.sample(frac=0.1, random_state=0).index, "AveOccup"] = np.nan

# 60% train, 20% dev, 20% test
train_set, test_set = train_test_split(housing_data, test_size=0.2, random_state=42)
train_set, dev_set = train_test_split(train_set, test_size=0.25, random_state=42)

# carving and saving
features = Features(numericals=["MedInc", "HouseAge", "AveRooms", "AveBedrms", "Population", "AveOccup"])
auto_carver = ContinuousCarver(features=features, min_freq=0.1, max_n_mod=4)
auto_carver.fit(train_set, train_set[target], X_dev=dev_set, y_dev=dev_set[target])
auto_carver.save(Path(tempfile.gettempdir()) / "continuous_carver.json")

## Loading a saved carver

Any saved carver can be loaded and edited.

In [3]:
from AutoCarver import ContinuousCarver

auto_carver = ContinuousCarver.load(Path(tempfile.gettempdir()) / "continuous_carver.json")
feature = auto_carver.features["Population"]
auto_carver.summary.loc[[str(feature)]]

content  \
feature                 kruskal  n_mod label                         
Numerical('Population') 12.60135 4     0          (-inf, 6.24e+02]   
                                       1      (6.24e+02, 7.90e+02]   
                                       2      (7.90e+02, 1.01e+03]   
                                       3           (1.01e+03, inf)   

                                              target_mean  frequency   count  \
feature                 kruskal  n_mod label                                   
Numerical('Population') 12.60135 4     0         2.107805   0.150436  1863.0   
                                       1         2.177366   0.099725  1235.0   
                                       2         2.155417   0.149871  1856.0   
                                       3         2.018133   0.599968  7430.0   

                                                   std  dropped dropped_reason  
feature                 kruskal  n_mod label                                    
Numerical('Population') 12.60135 4     0      1.316707    False           None  
                                       1      1.272533    False           None  
                                       2      1.232086    False           None  
                                       3      1.057706    False           None

``Population`` is carved into four bins. Its mean house values (2.11, 2.18, 2.16 and 2.02) are close to each other: as in the Continuous Regression example, this feature carries little signal. Editing its bins by hand changes little for a model, which makes it a good feature to practise on.

# Updating/Adjusting AutoCarver
## Updating group thresholds/values

Let's say one wants to adjust the values of a carved quantitative feature for better readability. Take feature ``"Population"`` for example, one could prefer group thresholds rounded to the nearest ten.

In [4]:
print(f"Upper thresholds for groups of feature {feature}: {list(feature.values)}")

Upper thresholds for groups of feature Numerical('Population'): [624.0, 790.0, 1011.0, inf]


In [5]:
# rounding each group's upper boundary to the closest ten
# (the last bin ends at +inf: it has no boundary to move)
for label, threshold in zip(list(feature.labels), list(feature.values)):
    if threshold != np.inf:
        feature.set_boundary(label, 10 * round(threshold / 10, 0))

auto_carver.summary.loc[[str(feature)]]

content  \
feature                 kruskal  n_mod label                         
Numerical('Population') 12.60135 4     0          (-inf, 6.20e+02]   
                                       1      (6.20e+02, 7.90e+02]   
                                       2      (7.90e+02, 1.01e+03]   
                                       3           (1.01e+03, inf)   

                                              target_mean  frequency  count  \
feature                 kruskal  n_mod label                                  
Numerical('Population') 12.60135 4     0              NaN        NaN    NaN   
                                       1              NaN        NaN    NaN   
                                       2              NaN        NaN    NaN   
                                       3              NaN        NaN    NaN   

                                              std  dropped dropped_reason  
feature                 kruskal  n_mod label                               
Numerical('Population') 12.60135 4     0      NaN    False           None  
                                       1      NaN    False           None  
                                       2      NaN    False           None  
                                       3      NaN    False           None

Using ``NumericalFeature.set_boundary()`` we rounded the thresholds of our groups: 624 became 620 and 1011 became 1010, while 790 was already round.

Be careful with changes to **Carvers** as no viability checks are performed: the statistics of every bin touched by an edit are reset to ``NaN``, since their true value is unknowable without a refit. Here every bin was touched, so all statistics are ``NaN``. The ``kruskal`` and ``n_mod`` index levels still show the values found at fit time.

## Grouping built modalities

Let's say one thinks there are too many modalities for carved feature ``"Population"``. Using ``BaseFeature.group()`` we can group two adjacent bins.

In [6]:
print(f"Upper thresholds for groups of feature {feature}: {list(feature.values)}")

Upper thresholds for groups of feature Numerical('Population'): [620.0, 790.0, 1010.0, inf]


The first two upper bounds (620 and 790) are close to one another compared to the last one. Let's group the two bins they delimit, by discarding the first label into the second:

In [7]:
labels = list(feature.labels)

# grouping the first bin into the second: only adjacent bins may be grouped,
# otherwise labels and discretizations would be wrong
feature.group([labels[0]], labels[1])

auto_carver.summary.loc[[str(feature)]]

content  \
feature                 kruskal  n_mod label                         
Numerical('Population') 12.60135 4     0          (-inf, 7.90e+02]   
                                       1      (7.90e+02, 1.01e+03]   
                                       2           (1.01e+03, inf)   

                                              target_mean  frequency  count  \
feature                 kruskal  n_mod label                                  
Numerical('Population') 12.60135 4     0              NaN        NaN    NaN   
                                       1              NaN        NaN    NaN   
                                       2              NaN        NaN    NaN   

                                              std  dropped dropped_reason  
feature                 kruskal  n_mod label                               
Numerical('Population') 12.60135 4     0      NaN    False           None  
                                       1      NaN    False           None  
                                       2      NaN    False           None

The two first bins now form a single bin, ``(-inf, 7.90e+02]``, and the labels were renumbered from ``0`` to ``2``.

## Missing values

The carver saw missing values of ``AveOccup`` at fit time, so it placed them like any other modality, alone or grouped with a bin. The summary shows them as ``__NAN__`` in the ``content`` column.

In [8]:
auto_carver.summary.loc[[str(auto_carver.features["AveOccup"])]]

content  \
feature               kruskal    n_mod label                                  
Numerical('AveOccup') 815.930303 4     0                   (-inf, 2.26e+00]   
                                       1               (2.26e+00, 3.10e+00]   
                                       2      (3.10e+00, 3.54e+00], __NAN__   
                                       3                    (3.54e+00, inf)   

                                              target_mean  frequency   count  \
feature               kruskal    n_mod label                                   
Numerical('AveOccup') 815.930303 4     0         2.548231   0.149629  1853.0   
                                       1         2.169368   0.448643  5556.0   
                                       2         1.943154   0.252180  3123.0   
                                       3         1.494448   0.149548  1852.0   

                                                   std  dropped dropped_reason  
feature               kruskal    n_mod label                                    
Numerical('AveOccup') 815.930303 4     0      1.306124    False           None  
                                       1      1.175204    False           None  
                                       2      1.057386    False           None  
                                       3      0.712437    False           None

Missing values of ``AveOccup`` were grouped with label ``2`` (``content="(3.10e+00, 3.54e+00], __NAN__"``), next to households of 3.1 to 3.54 people.

Let's check where they go when transforming the test set:

In [9]:
import pandas as pd

test_set_processed = auto_carver.transform(test_set)
pd.crosstab(test_set["AveOccup"].isna(), test_set_processed["AveOccup"], rownames=["missing"])

AveOccup,0,1,2,3
missing,,,,
False,607,1923,563,648
True,0,0,387,0


All 387 missing values of the test set go to label ``2``, like the households of 3.1 to 3.54 people.

Missing values must be present at fit time for the carver to place them. ``MedInc`` had none, so missing values in new data are rejected by ``transform``:

In [10]:
# inserting missing values into a feature that had none at fit time
test_set_with_nan = test_set.copy()
test_set_with_nan.loc[test_set_with_nan.sample(frac=0.1, random_state=0).index, "MedInc"] = np.nan

try:
    auto_carver.transform(test_set_with_nan)
except ValueError as error:
    print(error)

[Numerical('MedInc')] Unexpected NaNs.


The editing methods (``group``, ``set_boundary``, ``split``) work on the bins found at fit time. They cannot add a bin for missing values, nor take missing values out of a numerical bin. If missing values can appear in production, they need to be in the train and dev samples when the carver is fitted.

## What's next?

* See [Features](../../../features.rst) for the full list of editing methods, including ``move`` and ``ungroup`` for qualitative features.